# AgentKV cloud pilot
Select a GPU runtime. This notebook downloads the pinned Qwen3-0.6B checkpoint in the cloud only. Upload the prepared source zip from your laptop and extract it below. All method code lives in the repository; this notebook only runs it. GPU quality/performance results are not yet verified.
On Colab use the Files panel to upload `agentkv-source.zip` into `/content`. On another service change SOURCE_ZIP and WORKDIR. Save the exported results before ending the session.

In [ ]:
from pathlib import Path
import os, zipfile
SOURCE_ZIP = Path('/content/agentkv-source.zip')
WORKDIR = Path('/content/agentkv-work')
WORKDIR.mkdir(exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    archive.extractall(WORKDIR)
os.chdir(WORKDIR)
print(Path.cwd())


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before proceeding.'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print(torch.cuda.get_device_name(0))
print('GPU memory GB:', torch.cuda.get_device_properties(0).total_memory / 1e9)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'Attention/requirements-agentkv.txt'], check=True)


If this runtime already imported Transformers or NumPy before installation, restart the Python session, set the working directory again, then continue. Do not replace the runtime's CUDA PyTorch with a CPU wheel.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', 'Attention/tests', '-q'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'Attention.experiments.agentkv.run', '--config', 'Attention/configs/agentkv/pilot.json', '--out', 'artifacts/agentkv-pilot'], check=True)
# For interrupted runs with unchanged code/config/runtime, add --resume.


In [ ]:
import json
from pathlib import Path
summary = json.loads(Path('artifacts/agentkv-pilot/summary.json').read_text())
for row in summary:
    print(row['method'], row['policy'], 'quality=', row.get('quality_value'), 'tokens/s=', row.get('throughput'), 'failures=', row.get('failures'))


In [ ]:
import shutil
result_zip = shutil.make_archive('/content/agentkv-results', 'zip', 'artifacts/agentkv-pilot')
print('Download:', result_zip)
# Colab convenience (otherwise use the runtime file browser):
try:
    from google.colab import files
    files.download(result_zip)
except ImportError:
    pass
